In [19]:
!nvidia-smi

Sat Dec 13 05:08:57 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.06             Driver Version: 570.124.06     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:01:00.0  On |                  Off |
| 30%   60C    P0             71W /  300W |   16098MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
import os, tempfile, pathlib

print("pwd：", os.getcwd())
print("current home：", pathlib.Path.home())
print("tmp folder：", tempfile.gettempdir())

os.environ["PYCOX_DATA_DIR"] = "./data" 

import pathlib
pathlib.Path("./data").mkdir(exist_ok=True)

pwd： /home/UT_shared/data
current home： /home/jiahuiflora
tmp folder： /tmp


In [21]:
import numpy as np
import pandas as pd
import torch
import torchtuples as tt
from torch import nn

from einops import rearrange, repeat
from pycox.preprocessing.label_transforms import LabTransDiscreteTime
from pycox.models import DeepHit
from pycox.evaluation import EvalSurv
from tab_transformer_pytorch import TabTransformer as BaseTabTransformer
from sklearn.model_selection import train_test_split

SEED = 1234
np.random.seed(SEED)
torch.manual_seed(SEED)

In [22]:
import scipy.integrate
from scipy.integrate import simpson

scipy.integrate.simps = simpson 

In [23]:
import argparse
parser = argparse.ArgumentParser()
parser.add_argument("--lr", type=float, choices=[1e-3, 5e-3, 1e-4], default=1e-3)
parser.add_argument("--decoupled_weight_decay", type=float, choices=[0.01, 0.02], default=0.01)
parser.add_argument("--cycle_eta_multiplier", type=float, choices=[0.8, 0.7], default=0.8)
parser.add_argument("--alpha", type=float, choices=[0.2, 0.3], default=0.2)
parser.add_argument("--batch_size", type=int, choices=[64, 128, 256], default=64)
parser.add_argument("--dropout", type=float, choices=[0.1, 0.2, 0.3], default=0.1)
parser.add_argument("--num_epochs", type=int, choices=[5, 50, 100, 150], default=5)
parser.add_argument("--num_durations", type=int, choices=[10, 12, 15, 20, 30], default=10)
parser.add_argument("--embed_dim", type=int, choices=[200, 500, 1000, 2000], default=500)
args = parser.parse_args(args=[])

In [24]:
os.chdir('/home/UHN')
dict_df = pd.read_csv('Data dictionary_for_UofT_final.csv')

os.chdir('/home/UT_shared/data')
df = pd.read_csv("df_imputed.csv")

df_train, df_rest = train_test_split(
    df,
    test_size=0.3,           
    random_state=SEED
)

df_val, df_temp = train_test_split(
    df_rest,
    test_size=2/3,           
    random_state=SEED
)

df_test, df_cal = train_test_split(
    df_temp,
    test_size=1/2,           
    random_state=SEED
)

print(df_train.shape, df_val.shape, df_test.shape, df_cal.shape)

# df_train.to_csv("/home/UT_shared/data/train.csv", index=False)
# df_val.to_csv("/home/UT_shared/data/val.csv", index=False)
# df_test.to_csv("/home/UT_shared/data/test.csv", index=False)
# df_cal.to_csv("/home/UT_shared/data/cal.csv", index=False)


(74655, 80) (10665, 80) (10665, 80) (10666, 80)


In [25]:
column_names_index = df.columns
print(column_names_index)

Index(['demographics_age_index_ecg', 'hypertension_icd10', 'diabetes_combined',
       'dyslipidemia_combined', 'dcm_icd10', 'hcm_icd10',
       'myocarditis_icd10_prior', 'pericarditis_icd10_prior',
       'aortic_aneurysm_icd10', 'aortic_dissection_icd10_prior',
       'pulmonary_htn_icd10', 'amyloid_icd10', 'copd_icd10',
       'obstructive._sleep_apnea_icd10', 'hyperthyroid_icd10',
       'hypothyroid_icd10', 'rheumatoid_arthritis_icd10', 'sle_icd10',
       'sarcoid_icd10', 'cancer_any_icd10',
       'event_cv_hf_admission_icd10_prior', 'event_cv_ep_vt_any_icd10_prior',
       'event_cv_ep_sca_survived_icd10_cci_prior',
       'event_cv_cns_stroke_ischemic_icd10_prior',
       'event_cv_cns_stroke_hemorrh_icd10_prior',
       'event_cv_cns_tia_icd10_prior', 'pci_prior', 'cabg_prior',
       'transplant_heart_cci_prior', 'lvad_cci_prior',
       'pacemaker_permanent_cci_prior', 'crt_cci_prior', 'icd_cci_prior',
       'ecg_resting_hr', 'ecg_resting_pr', 'ecg_resting_qrs',
       'e

In [7]:
# dict_df.loc[15, "Variable name"]
dict_df.loc[15, "Variable name"] = "obstructive._sleep_apnea_icd10"

categ_vars = dict_df.loc[
    dict_df["Variable type"].isin(["boolean", "categorical"]),
    "Variable name"
].tolist()

cont_vars = dict_df.loc[
    dict_df["Variable type"] == "numeric",
    "Variable name"
].tolist()

categ_idx_1 = [df.columns.get_loc(v) for v in categ_vars if v in df.columns]
cont_idx  = [df.columns.get_loc(v) for v in cont_vars if v in df.columns]

print(categ_idx_1)
print(cont_idx)

vars_to_add = ['sex_imp', 'acei_arb_entresto', 'acute_mi_angina_other']

categ_idx = categ_idx_1.copy()

for v in vars_to_add:
    if v in df.columns:
        idx = df.columns.get_loc(v)
        if idx not in categ_idx:   # avoid duplicates
            categ_idx.append(idx)

print("Updated categorical idx:", categ_idx)

only_in_old = sorted(set(categ_idx_1) - set(categ_idx))  
only_in_new = sorted(set(categ_idx) - set(categ_idx_1))  

print("Idx only in original (removed):", only_in_old)
print("Idx only in updated (added):", only_in_new)

all_used_idx = set(cont_idx) | set(categ_idx)
other_idx = [i for i in range(len(df.columns)) if i not in all_used_idx]
other_vars = df.columns[other_idx].tolist()

print("Other variable indices:", other_idx)
print("Other variables:", other_vars)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74]
[0, 33, 34, 35, 36, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
Updated categorical idx: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 79, 75, 76]
Idx only in original (removed): []
Idx only in updated (added): [75, 76, 79]
Other variable indices: [77, 78]
Other variables: ['status', 'year']


In [8]:
x_cat_train = torch.tensor(df_train.iloc[:, categ_idx].to_numpy(), dtype=torch.long)
x_cont_train = torch.tensor(df_train.iloc[:, cont_idx].to_numpy(), dtype=torch.float32)

x_cat_val = torch.tensor(df_val.iloc[:, categ_idx].to_numpy(), dtype=torch.long)
x_cont_val = torch.tensor(df_val.iloc[:, cont_idx].to_numpy(), dtype=torch.float32)

x_cat_test = torch.tensor(df_test.iloc[:, categ_idx].to_numpy(), dtype=torch.long)
x_cont_test = torch.tensor(df_test.iloc[:, cont_idx].to_numpy(), dtype=torch.float32)

In [9]:
get_target = lambda df: (df['year'].values, df['status'].values)

class LabTransform(LabTransDiscreteTime):
    def transform(self, durations, events):
        durations, is_event = super().transform(durations, events > 0)
        events[is_event == 0] = 0
        return durations, events.astype('int64')

In [10]:
labtrans = LabTransform(args.num_durations)
y_train = labtrans.fit_transform(*get_target(df_train))
y_val = labtrans.transform(*get_target(df_val))
y_train = tuple(torch.tensor(arr, dtype=torch.long) for arr in y_train)
y_val = tuple(torch.tensor(arr, dtype=torch.long) for arr in y_val)

In [11]:
durations_test, events_test = get_target(df_test)
duration_idx = labtrans.cuts

def exists(val):
    return val is not None

def default(val, d):
    return val if exists(val) else d

class TabTransformerwithEmbedding(BaseTabTransformer):
    def forward(self, x_categ, x_cont, return_attn=False, return_embedding=False):
        xs = []

        assert x_categ.shape[-1] == self.num_categories
        if self.num_unique_categories > 0:
            x_categ = x_categ + self.categories_offset
            categ_embed = self.category_embed(x_categ)

            if self.use_shared_categ_embed:
                shared_categ_embed = repeat(self.shared_category_embed, 'n d -> b n d', b=categ_embed.shape[0])
                categ_embed = torch.cat((categ_embed, shared_categ_embed), dim=-1)

            x, attns = self.transformer(categ_embed, return_attn=True)
            flat_categ = rearrange(x, 'b ... -> b (...)')
            xs.append(flat_categ)

        if self.num_continuous > 0:
            if exists(self.continuous_mean_std):
                mean, std = self.continuous_mean_std.unbind(dim=-1)
                x_cont = (x_cont - mean) / std
            normed_cont = self.norm(x_cont)
            xs.append(normed_cont)

        x = torch.cat(xs, dim=-1)
        x = self.embedding_proj(x)

        if return_embedding:
            if return_attn:
                return x, attns
            return x

        logits = self.mlp(x)
        if return_attn:
            return logits, attns
        return logits

class TabTransformerSharedCauseSpecificNet(nn.Module):
    def __init__(self, tab_config, embed_dim, num_nodes_shared, num_nodes_indiv, num_risks, out_features, dropout=0.1):
        super().__init__()
        self.tab = TabTransformerwithEmbedding(**tab_config)
        self.shared_mlp = tt.practical.MLPVanilla(embed_dim, num_nodes_shared[:-1], num_nodes_shared[-1],
                                                  batch_norm=True, dropout=dropout)
        self.risk_nets = nn.ModuleList([
            tt.practical.MLPVanilla(num_nodes_shared[-1], num_nodes_indiv, out_features,
                                    batch_norm=True, dropout=dropout)
            for _ in range(num_risks)
        ])

    def forward(self, x_cat, x_cont):
        x_embed = self.tab(x_cat, x_cont, return_embedding=True)  # shape: [batch, embed_dim]
        shared_out = self.shared_mlp(x_embed)  # shape: [batch, hidden]

        out = [risk_net(shared_out) for risk_net in self.risk_nets]  # List of [batch, out_features]
        return torch.stack(out, dim=1)  # shape: [batch, num_risks, out_features]

categories = [int(df_train.iloc[:, idx].max()) + 1 for idx in categ_idx]

In [12]:
tab_config = {
    'categories': categories,
    'num_continuous': len(cont_idx),
    'dim': 32,
    'depth': 6,
    'heads': 8,
    'attn_dropout': args.dropout,
    'ff_dropout': args.dropout,
    'mlp_hidden_mults': (4, 2),
    'mlp_act': nn.ReLU(),
    'dim_out': 100
}

embed_dim = args.embed_dim
in_features = args.embed_dim
num_nodes = [128, 64]
num_risks = y_train[1].max()
out_features = len(labtrans.cuts)

net = TabTransformerSharedCauseSpecificNet(
    tab_config=tab_config,
    embed_dim=embed_dim,
    num_nodes_shared=[64, 64],
    num_nodes_indiv=[32],
    num_risks=y_train[1].max(),
    out_features=len(labtrans.cuts),
    dropout=args.dropout
)

In [13]:
# optimizer = tt.optim.AdamWR(lr=args.lr)
optimizer = tt.optim.AdamWR(lr=args.lr, decoupled_weight_decay=args.decoupled_weight_decay, cycle_eta_multiplier=args.cycle_eta_multiplier)
model = DeepHit(net, optimizer, alpha=args.alpha, sigma=0.1, duration_index=duration_idx)

In [14]:
model.fit((x_cat_train, x_cont_train), y_train,
          batch_size=args.batch_size,
          epochs=args.num_epochs,
          callbacks=[tt.callbacks.EarlyStopping(patience=1)],
          val_data=((x_cat_val, x_cont_val), y_val),
          verbose=True)

/opt/conda/lib/python3.13/site-packages/torchtuples/callbacks.py:607: UserWarning: This overload of add is deprecated:
	add(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add(Tensor other, *, Number alpha = 1) (Triggered internally at /pytorch/torch/csrc/utils/python_arg_parser.cpp:1805.)
  p.data = p.data.add(-weight_decay * eta, p.data)


0:	[26s / 26s],		train_loss: 0.2498,	val_loss: 0.2219
1:	[25s / 52s],		train_loss: 0.2267,	val_loss: 0.2171
2:	[25s / 1m:17s],		train_loss: 0.2131,	val_loss: 0.2041
3:	[25s / 1m:43s],		train_loss: 0.2145,	val_loss: 0.2926


In [26]:
surv = model.predict_surv_df((x_cat_test, x_cont_test))
ev = EvalSurv(surv, durations_test, events_test != 0, censor_surv='km')

c_index = ev.concordance_td()
time_grid = np.unique(durations_test)
ibs = ev.integrated_brier_score(time_grid)
# model.save_net('/home/UT_shared/result/500_embedding_duration_10_deephit_tabtransformer_shared_node_epoch50.pt')

In [27]:
print(f"Hyperparameters – lr: {args.lr}, wd: {args.decoupled_weight_decay}, eta: {args.cycle_eta_multiplier}, "
      f"alpha: {args.alpha}, batch_size: {args.batch_size}, dropout: {args.dropout}, "
      f"epochs: {args.num_epochs}, durations: {args.num_durations}, embed_dim: {args.embed_dim} | "
      f"C-index: {c_index:.4f}, IBS: {ibs:.4f}")


Hyperparameters – lr: 0.001, wd: 0.01, eta: 0.8, alpha: 0.2, batch_size: 64, dropout: 0.1, epochs: 5, durations: 10, embed_dim: 500 | C-index: 0.7976, IBS: 0.0928


In [28]:
model_dir = 'end_to_end'
os.makedirs(model_dir, exist_ok=True)

save_path = os.path.join(
    model_dir,
    (f"deephit74"
     f"-lr{args.lr}"
     f"-wd{args.decoupled_weight_decay}"
     f"-eta{args.cycle_eta_multiplier}"
     f"-alpha{args.alpha}"
     f"-bs{args.batch_size}"
     f"-drop{args.dropout}"
     f"-epochs{args.num_epochs}"
     f"-dur{args.num_durations}"
     f"-embed{args.embed_dim}.pt")
)


In [29]:
import gc     
try:
    del net      # delete model
except NameError:
    pass

try:
    del trainer  # delete trainer or training helper
except NameError:
    pass

try:
    del loss     # delete loss tensor/variable
except NameError:
    pass

try:
    del data     # delete dataset or data loader
except NameError:
    pass

torch.cuda.empty_cache()  # release all unused cached GPU memory

# Force Python garbage collection
gc.collect()  # free up unreferenced Python objects

print("Cleanup complete")


Cleanup complete


In [30]:
!nvidia-smi

Sat Dec 13 05:22:59 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.06             Driver Version: 570.124.06     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:01:00.0  On |                  Off |
| 32%   62C    P3             71W /  300W |    1131MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import torch
import torchtuples as tt
from torch import nn
from tab_transformer_pytorch import TabTransformer as BaseTabTransformer

class TabTransformerwithEmbedding(BaseTabTransformer):
    def forward(self, x_categ, x_cont, return_attn=False, return_embedding=False):
        xs = []

        assert x_categ.shape[-1] == self.num_categories
        if self.num_unique_categories > 0:
            x_categ = x_categ + self.categories_offset
            categ_embed = self.category_embed(x_categ)

            if self.use_shared_categ_embed:
                shared_categ_embed = repeat(self.shared_category_embed, 'n d -> b n d', b=categ_embed.shape[0])
                categ_embed = torch.cat((categ_embed, shared_categ_embed), dim=-1)

            x, attns = self.transformer(categ_embed, return_attn=True)
            flat_categ = rearrange(x, 'b ... -> b (...)')
            xs.append(flat_categ)

        if self.num_continuous > 0:
            if exists(self.continuous_mean_std):
                mean, std = self.continuous_mean_std.unbind(dim=-1)
                x_cont = (x_cont - mean) / std
            normed_cont = self.norm(x_cont)
            xs.append(normed_cont)

        x = torch.cat(xs, dim=-1)
        x = self.embedding_proj(x)

        if return_embedding:
            if return_attn:
                return x, attns
            return x

        logits = self.mlp(x)
        if return_attn:
            return logits, attns
        return logits

class TabTransformerSharedCauseSpecificNet(nn.Module):
    def __init__(self, tab_config, embed_dim, num_nodes_shared, num_nodes_indiv, num_risks, out_features, dropout=0.1):
        super().__init__()
        self.tab = TabTransformerwithEmbedding(**tab_config)
        self.shared_mlp = tt.practical.MLPVanilla(embed_dim, num_nodes_shared[:-1], num_nodes_shared[-1],
                                                  batch_norm=True, dropout=dropout)
        self.risk_nets = nn.ModuleList([
            tt.practical.MLPVanilla(num_nodes_shared[-1], num_nodes_indiv, out_features,
                                    batch_norm=True, dropout=dropout)
            for _ in range(num_risks)
        ])

    def forward(self, x_cat, x_cont):
        x_embed = self.tab(x_cat, x_cont, return_embedding=True)  # shape: [batch, embed_dim]
        shared_out = self.shared_mlp(x_embed)  # shape: [batch, hidden]

        out = [risk_net(shared_out) for risk_net in self.risk_nets]  # List of [batch, out_features]
        return torch.stack(out, dim=1)  # shape: [batch, num_risks, out_features]

# 2. Load the file using the security bypass (weights_only=False)
model_path = "/home/UT_shared/final_model/end-lr0.0001-wd0.01-eta0.8-alpha0.2-bs256-drop0.1-epochs100-dur30-embed500.pt"

# We use a lower-level loading method to see if we can peek at the keys 
# without triggering the class reconstruction that is failing.
try:
    # Attempt to load and catch the dictionary directly
    checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)
    
    if isinstance(checkpoint, dict):
        print("Successfully loaded as a dictionary!")
        for key in checkpoint.keys():
            print(f"Key: {key}")
    else:
        print(f"Loaded object type: {type(checkpoint)}")
        # If it's the model object itself, we can still list parameters
        for name, param in checkpoint.named_parameters():
            print(f"Parameter: {name} | Shape: {list(param.shape)}")

except AttributeError as e:
    print(f"Direct load failed with: {e}")
    print("\nAttempting to load weights by stripping class requirements...")
    
    # Alternative: Load only the state_dict if the file contains the full model
    # Note: This is a 'hack' to see the data when the class definition is broken
    import pickle
    
    class IdentityUnpickler(pickle.Unpickler):
        def find_class(self, module, name):
            if name == 'Transformer':
                # Map the missing attribute to a basic nn.Module to bypass the error
                return torch.nn.Module
            return super().find_class(module, name)

    with open(model_path, 'rb') as f:
        unpickler = IdentityUnpickler(f)
        try:
            checkpoint = unpickler.load()
            print("Successfully bypassed AttributeError. Parameter keys:")
            # If it's a model object, get the state_dict
            state = checkpoint.state_dict() if hasattr(checkpoint, 'state_dict') else checkpoint
            for k in state.keys():
                print(k)
        except Exception as e2:
            print(f"Bypass also failed: {e2}")

Direct load failed with: Can't get attribute 'Transformer' on <module 'tab_transformer_pytorch' from '/opt/conda/lib/python3.13/site-packages/tab_transformer_pytorch/__init__.py'>

Attempting to load weights by stripping class requirements...
Bypass also failed: persistent IDs in protocol 0 must be ASCII strings


In [9]:
import torch
import tab_transformer_pytorch
from torch import nn
import torchtuples as tt

# 1. Hoist the internal classes to the top level as you identified
from tab_transformer_pytorch.tab_transformer_pytorch import (
    Transformer as InnerTransformer,
    PreNorm as InnerPreNorm,
    Attention as InnerAttention,
    FeedForward as InnerFeedForward,
    GEGLU as InnerGEGLU,
    MLP as InnerMLP,
)

setattr(tab_transformer_pytorch, "Transformer", InnerTransformer)
setattr(tab_transformer_pytorch, "PreNorm", InnerPreNorm)
setattr(tab_transformer_pytorch, "Attention", InnerAttention)
setattr(tab_transformer_pytorch, "FeedForward", InnerFeedForward)
setattr(tab_transformer_pytorch, "GEGLU", InnerGEGLU)
setattr(tab_transformer_pytorch, "MLP", InnerMLP)

# 2. Define your training classes so the unpickler recognizes them
from tab_transformer_pytorch import TabTransformer as BaseTabTransformer

class TabTransformerwithEmbedding(BaseTabTransformer):
    def forward(self, x_categ, x_cont, return_attn=False, return_embedding=False):
        pass 

class TabTransformerSharedCauseSpecificNet(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()

# 3. Load the model
model_path = "/home/UT_shared/final_model/end-lr0.0001-wd0.01-eta0.8-alpha0.2-bs256-drop0.1-epochs100-dur30-embed500.pt"

try:
    # Use weights_only=False because we are reconstructing custom classes
    model_obj = torch.load(model_path, map_location='cpu', weights_only=False)
    
    # Get the state_dict (the actual weights)
    if hasattr(model_obj, 'state_dict'):
        state_dict = model_obj.state_dict()
    else:
        state_dict = model_obj

    print("✅ Model loaded successfully!\n")
    print(f"{'Layer Name':<65} | {'Shape'}")
    print("-" * 85)
    
    for name, param in state_dict.items():
        if torch.is_tensor(param):
            print(f"{name:<65} | {list(param.shape)}")

except Exception as e:
    print(f"❌ Load failed: {e}")

✅ Model loaded successfully!

Layer Name                                                        | Shape
-------------------------------------------------------------------------------------
tab.shared_category_embed                                         | [61, 4]
tab.categories_offset                                             | [61]
tab.category_embed.weight                                         | [124, 28]
tab.norm.weight                                                   | [17]
tab.norm.bias                                                     | [17]
tab.transformer.layers.0.0.static_alpha                           | [4, 5]
tab.transformer.layers.0.0.dynamic_alpha_fn                       | [32, 5]
tab.transformer.layers.0.0.dynamic_alpha_scale                    | []
tab.transformer.layers.0.0.static_beta                            | [4]
tab.transformer.layers.0.0.dynamic_beta_fn                        | [32]
tab.transformer.layers.0.0.dynamic_beta_scale                     | []

In [10]:
import pandas as pd

# Let's say you want to look at the embedding projection weights
weights = state_dict['tab.embedding_proj.weight'].numpy()
df_weights = pd.DataFrame(weights)

# Save to CSV to inspect in Excel
df_weights.to_csv("embedding_weights.csv", index=False)
print("Saved embedding weights to embedding_weights.csv")

Saved embedding weights to embedding_weights.csv
